## 1.获取数据

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. 准备+解压训练数据

In [ ]:

import yaml
import os

!pip install ultralytics -q
import ultralytics
from ultralytics import YOLO

#  定义路径
zip_path = "/content/drive/MyDrive/Colab Notebooks/DSAIproject/SelfDrivingCar_v3_fixed_small.yolov8.zip"
extract_path = "/content/dataset"

#  清理旧文件夹并解压
!rm -rf {extract_path}
!mkdir -p {extract_path}
print("📦 正在极速解压数据到本地盘...")
!unzip -q "{zip_path}" -d {extract_path}

print(f"✅ 解压完成！数据路径: {extract_path}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 75.9 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
📦 正在极速解压数据到本地盘...
✅ 解压完成！数据路径: /content/dataset


In [ ]:
# --- 1. 配置基础路径 ---
# 确保这个路径和你 Cell 2 解压的路径一致
dataset_root = "/content/dataset"
yaml_path = os.path.join(dataset_root, "data.yaml")
# 根据你之前的截图，图片实际存放在 export/images
images_dir = "export/images"

# --- 2. 读取并修改 YAML 内容 ---
if os.path.exists(yaml_path):
    with open(yaml_path, 'r') as f:
        # 使用 Loader 确保安全读取
        config = yaml.safe_load(f)

    # 核心修正：设置根目录，并将所有阶段都指向现有的图片文件夹
    # 这样 Baseline 测试才能在没有拆分数据集的情况下跑通
    config['path'] = dataset_root
    config['train'] = images_dir
    config['val'] = images_dir
    config['test'] = images_dir

    # --- 3. 写回修改后的内容 ---
    with open(yaml_path, 'w') as f:
        yaml.dump(config, f, sort_keys=False)

    print("✅ Cell 3 任务完成：data.yaml 已成功修正！")
    print(f"📍 根路径 (path): {config['path']}")
    print(f"🖼️ 目标图片目录: {images_dir}")
    print(f"👥 识别类别 ({config['nc']}类): {config['names']}")
else:
    print(f"❌ 错误：在 {yaml_path} 找不到 data.yaml，请检查 Cell 2 是否运行成功。")

# --- 4. 验证图片路径是否存在 ---
full_images_path = os.path.join(dataset_root, images_dir)
if os.path.exists(full_images_path):
    img_count = len([f for f in os.listdir(full_images_path) if f.endswith(('.jpg', '.jpeg', '.png'))])
    print(f"🔍 路径验证成功！共发现 {img_count} 张图片。")
else:
    print(f"⚠️ 警告：路径 {full_images_path} 不存在，请检查解压后的文件夹结构。")

✅ Cell 3 任务完成：data.yaml 已成功修正！
📍 根路径 (path): /content/dataset
🖼️ 目标图片目录: export/images
👥 识别类别 (11类): ['biker', 'car', 'pedestrian', 'trafficLight', 'trafficLight-Green', 'trafficLight-GreenLeft', 'trafficLight-Red', 'trafficLight-RedLeft', 'trafficLight-Yellow', 'trafficLight-YellowLeft', 'truck']
🔍 路径验证成功！共发现 29800 张图片。


## 2.准备原始yolo v8n + Finetune10轮适配11类分类任务

In [ ]:
#cell4:安装与加载模型
# 1. 安装 Ultralytics 库 (YOLOv8 官方库)
# -q 表示静默安装，减少屏幕上的干扰信息


# 2. 检查安装是否成功
ultralytics.checks()

# 3. 加载预训练模型 YOLOv8n (Nano 版本)
# 这是官方在大型数据集上训练好的权重，是我们实验的“原始起点”
model = YOLO('yolov8n.pt')

print("\n✅ Cell 4 任务完成：实验环境已就绪！")
print("🚀 模型已加载：YOLOv8n (Nano) —— 准备进行 Baseline 测试。")

Ultralytics 8.4.24 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 45.8/112.6 GB disk)

✅ Cell 4 任务完成：实验环境已就绪！
🚀 模型已加载：YOLOv8n (Nano) —— 准备进行 Baseline 测试。


In [ ]:
#cell4.5 finetune原始模型，适应11类的新数据

import os
import shutil

# --- 1. 定义路径 ---
PROJECT_DIR = '/content/drive/MyDrive/Colab Notebooks/DSAIproject'
SAVE_NAME = 'finetuned_baseline'
FINAL_MODEL_PATH = os.path.join(PROJECT_DIR, 'yolov8n_finetune.pt')

# 确保项目目录存在
if not os.path.exists(PROJECT_DIR):
    os.makedirs(PROJECT_DIR)

# --- 2. 启动训练 ---
# 使用你在 Cell 4 加载的 model 对象
print(f"🚀 开始微调模型 (设备: cuda)...")
results = model.train(
    data=yaml_path,
    epochs=10,        # 建议先跑 10 轮观察效果
    imgsz=640,
    batch=32,
    device='cuda',
    project=PROJECT_DIR,
    name=SAVE_NAME,
    exist_ok=True     # 如果文件夹已存在则覆盖
)

# --- 3. 提取最优权重并重命名保存 ---
# 训练产生的最佳权重路径通常是：project/name/weights/best.pt
best_model_temp = os.path.join(PROJECT_DIR, SAVE_NAME, 'weights', 'best.pt')

if os.path.exists(best_model_temp):
    # 将 best.pt 复制并重命名为 yolov8n_finetune.pt
    shutil.copy(best_model_temp, FINAL_MODEL_PATH)
    print(f"✅ 微调完成！模型已保存至: {FINAL_MODEL_PATH}")
else:
    print("⚠️ 警告：未找到训练生成的 best.pt 文件，请检查训练日志。")

🚀 开始微调模型 (设备: cuda)...
Ultralytics 8.4.24 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=finetuned_baseline, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=Tr

In [ ]:
#cell6:可视化 Baseline 预测

import PIL.Image as Image
import random
import glob

# 1. 随机挑选 3 张图片进行测试
image_files = glob.glob("/content/dataset/export/images/*.jpg")
sample_images = random.sample(image_files, 3)

# 2. 运行推理 (Inference)
# conf=0.25 是置信度阈值，只有模型认为概率大于 25% 的目标才会被画出来

model = YOLO(model_path)

# 运行推理
results = model.predict(source=sample_images, conf=0.25, save=True)

# 3. 在 Colab 中直接显示结果图
print("\n👁️ 正在展示原始模型的“视角”：")
for i, r in enumerate(results):
    # 转换为 PIL 图片格式并显示
    im_array = r.plot()  # 自动画好框的数组
    im = Image.fromarray(im_array[..., ::-1])  # RGB 转换
    display(im)
    print(f"图 {i+1}: 识别到了 {len(r.boxes)} 个目标")

print("\n✅ Cell 6 任务完成：可视化展示已生成。")


## 3. YOLO模型瘦身

* 3.1 非结构化剪枝（Unstructured Pruning）：
该方法以单个权重为最小裁剪单元，将绝对值低于阈值的参数直接置零。由于其极高的灵活性，它对模型原始精度（mAP）的破坏最小。然而，其生成的稀疏矩阵在常规硬件（如通用 CPU/GPU）上无法跳过零值计算，因此在没有特定稀疏硬件支持的情况下，无法带来实际的推理加速。

* 3.2 Mask 结构化剪枝（Mask-based Structured Pruning）：

该方法利用 PyTorch 原生的 torch.nn.utils.prune 等 API，在通道（Channel）层面上评估重要性并生成二进制掩码（Mask）。尽管其剪枝逻辑具有“结构化”的特征，但它并未物理改变张量的形状。在底层，网络依然保留了完整的参数字典并依赖动态钩子（Hook）进行掩码乘法运算。这导致模型文件的体积不仅没有减小，甚至因额外掩码的引入而略有增加，在工程部署上属于“伪结构化加速”。

* 3.3 物理重构结构化剪枝（Physical Reconstruction Structured Pruning / Hard Pruning）：

作为本项目的核心突破点，该方法在剪枝后对 YOLOv8 的网络层进行了硬重构（In-place Reconstruction）。它物理切除了不重要的卷积通道，使得矩阵维度真实变小。这种方式保留了规则且连续的稠密矩阵（Dense Matrix），完美契合边缘端芯片对连续内存读取和密集矩阵乘法的硬件偏好，从而真正实现了模型体积的等比例缩小与帧率（FPS）的显著提升。

In [ ]:
# 加载微调好的模型
model = YOLO("/content/drive/MyDrive/Colab Notebooks/DSAIproject/yolov8n_finetune.pt")

# 打印模型概览
print(model.model)

## 3.1 非结构化剪枝



## 3.1.1 非结构化剪枝10%



In [ ]:
import torch
from torch.nn.utils import prune

def apply_pruning(model, pruning_amount):
    """
    model: 你的 yolov8n_finetune 模型
    pruning_amount: 剪枝比例 (如 0.1 代表 10%)
    """
    print(f"开始执行 {pruning_amount*100}% 比例的 L1 范数非结构化剪枝...")

    for name, module in model.model.named_modules():
        # 排除第 22 层 Detect 层，只剪卷积层
        if isinstance(module, torch.nn.Conv2d) and "model.22" not in name:
            # 执行剪枝
            prune.l1_unstructured(module, name='weight', amount=pruning_amount)
            # 永久化修改
            prune.remove(module, 'weight')

    # 计算实际稀疏度（验证是否真的剪掉了 10%）
    total_zeros = 0
    total_params = 0
    for param in model.model.parameters():
        total_zeros += torch.sum(param == 0).item()
        total_params += param.nelement()

    print(f"剪枝完成！当前模型全局稀疏度: {total_zeros/total_params:.2%}")
    return model



In [ ]:

# 自由修改这个变量：0.1 代表 10%，0.3 代表 30%
current_amount = 0.1

# 执行剪枝
pruned_model = apply_pruning(model, pruning_amount=current_amount)

# 保存剪枝后的权重
pruned_10_path = "/content/drive/MyDrive/Colab Notebooks/DSAIproject/yolov8n_finetune_pruned_10.pt"
pruned_model.save(pruned_10_path)
print(f"✅ 10% 剪枝模型已存档至: {pruned_10_path}")


## 3.1.2 非结构化剪枝30%


In [ ]:
clean_model = YOLO("/content/drive/MyDrive/Colab Notebooks/DSAIproject/yolov8n_finetune.pt")

# 2. 设定新的剪枝比例
current_amount_30 = 0.3

# 3. 对“干净”的模型执行手术
pruned_model_30 = apply_pruning(clean_model, pruning_amount=current_amount_30)

开始执行 30.0% 比例的 L1 范数非结构化剪枝...
剪枝完成！当前模型全局稀疏度: 22.41%


In [ ]:
# 1. 保存剪枝后的权重
pruned_30_path = "/content/drive/MyDrive/Colab Notebooks/DSAIproject/yolov8n_finetune_pruned_30.pt"
pruned_model_30.save(pruned_30_path)
print(f"✅ 30% 剪枝模型已存档至: {pruned_30_path}")


✅ 30% 剪枝模型已存档至: /content/drive/MyDrive/Colab Notebooks/DSAIproject/yolov8n_finetune_pruned_30.pt
Ultralytics 8.4.24 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,007,793 parameters, 0 gradients
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1037.6±402.9 MB/s, size: 38.3 KB)
val: Scanning /content/dataset/export/labels.cache... 29800 images, 3500 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 29800/29800 7.8Git/s 0.0s
val: /content/dataset/export/images/1478021875081281646_jpg.rf.bEZPhuyXU5hIovwQSTIp.jpg: 1 duplicate labels removed
val: /content/dataset/export/images/1478021875081281646_jpg.rf.e9552980cf8c6fef4aa02cb84c6364f5.jpg: 1 duplicate labels removed
val: /content/dataset/export/images/1478897760163798179_jpg.rf.5Pzrj3Eg3vZuyl7ztKAt.jpg: 1 duplicate labels removed
val: /content/dataset/export/images/1478897760163798179_jpg.rf.98623be50b02ff17d58f89fddf7a0c6c.jpg: 1 duplicate labels removed
val: /content/dataset/export/images/147

## 3.1.3 非结构化剪枝50%

In [ ]:
clean_model = YOLO("/content/drive/MyDrive/Colab Notebooks/DSAIproject/yolov8n_finetune.pt")

# 2. 设定新的剪枝比例
current_amount_50 = 0.5

# 3. 对“干净”的模型执行手术
pruned_model_50 = apply_pruning(clean_model, pruning_amount=current_amount_50)

开始执行 50.0% 比例的 L1 范数非结构化剪枝...
剪枝完成！当前模型全局稀疏度: 37.35%


In [ ]:
# 1. 保存剪枝后的权重
pruned_50_path = "/content/drive/MyDrive/Colab Notebooks/DSAIproject/yolov8n_finetune_pruned_50.pt"
pruned_model_50.save(pruned_50_path)
print(f"✅ 50% 剪枝模型已存档至: {pruned_50_path}")


✅ 50% 剪枝模型已存档至: /content/drive/MyDrive/Colab Notebooks/DSAIproject/yolov8n_finetune_pruned_50.pt


## 3.2 指定层位联动结构化剪枝

## 3.2.1 指定层位联动结构化剪枝10%

In [ ]:
!pip install torch_pruning

In [ ]:
import torch
import torch.nn as nn
from ultralytics import YOLO
import os
from copy import deepcopy

# 1. 定义物理重建函数 (必须包含在当前 Cell)
def create_pruned_conv(conv_layer, keep_indices, mode='out'):
    """修改卷积层：mode='out' 剪输出通道，mode='in' 剪输入通道"""
    if mode == 'out':
        new_conv = nn.Conv2d(conv_layer.in_channels, len(keep_indices), conv_layer.kernel_size,
                             conv_layer.stride, conv_layer.padding, bias=conv_layer.bias is not None)
        new_conv.weight.data = conv_layer.weight.data[keep_indices].clone()
        if conv_layer.bias is not None: new_conv.bias.data = conv_layer.bias.data[keep_indices].clone()
    else: # mode == 'in'
        new_conv = nn.Conv2d(len(keep_indices), conv_layer.out_channels, conv_layer.kernel_size,
                             conv_layer.stride, conv_layer.padding, bias=conv_layer.bias is not None)
        new_conv.weight.data = conv_layer.weight.data[:, keep_indices].clone()
        if conv_layer.bias is not None: new_conv.bias.data = conv_layer.bias.data.clone()
    return new_conv

def create_pruned_bn(bn_layer, keep_indices):
    new_bn = nn.BatchNorm2d(len(keep_indices), eps=bn_layer.eps, momentum=bn_layer.momentum)
    if bn_layer.affine:
        new_bn.weight.data = bn_layer.weight.data[keep_indices].clone()
        new_bn.bias.data = bn_layer.bias.data[keep_indices].clone()
    return new_bn

# 2. 执行联动剪枝逻辑
finetune_ckpt = "/content/drive/MyDrive/Colab Notebooks/DSAIproject/yolov8n_finetune.pt"
model = YOLO(finetune_ckpt)
m = model.model # 访问底层的 nn.Module
PRUNING_RATIO = 0.10

print("🚀 开始链式联动剪枝...")

# 我们只剪枝 Backbone 的前几个简单卷积层（Layer 1, 4, 6），这些层最容易产生 FPS 收益且不易崩溃
# 避开 Layer 0 (输入层) 和复杂的 C2f 模块
target_layers = [1, 4, 6]

for i in target_layers:
    layer = m.model[i]
    if hasattr(layer, 'conv') and hasattr(layer, 'bn'):
        total = layer.bn.num_features
        keep_count = int(total * (1 - PRUNING_RATIO))
        keep_idx = list(range(keep_count))

        print(f"正在处理第 {i} 层: 通道 {total} -> {keep_count}")

        # A. 剪掉当前层的输出 (Conv + BN)
        layer.conv = create_pruned_conv(layer.conv, keep_idx, mode='out')
        layer.bn = create_pruned_bn(layer.bn, keep_idx)

        # B. 【关键】寻找下一层并剪掉它的输入
        # 在 YOLOv8n 中，第 i 层的下一层通常是 i+1
        next_layer = m.model[i+1]

        # 如果下一层是普通的 Conv
        if hasattr(next_layer, 'conv'):
            next_layer.conv = create_pruned_conv(next_layer.conv, keep_idx, mode='in')
            print(f"  --> 已同步修改第 {i+1} 层的输入端")
        # 如果下一层是 C2f 模块
        elif next_layer.__class__.__name__ == 'C2f':
            next_layer.cv1.conv = create_pruned_conv(next_layer.cv1.conv, keep_idx, mode='in')
            print(f"  --> 已同步修改第 {i+1} 层 (C2f) 的输入端")

# 3. 保存并重命名
save_path = "/content/drive/MyDrive/Colab Notebooks/DSAIproject/yolov8n_struct_pruned_10.pt"
model.save(save_path)
print(f"✅ 10% 结构化剪枝模型已保存: {save_path}")

🚀 开始链式联动剪枝...
正在处理第 1 层: 通道 32 -> 28
  --> 已同步修改第 2 层 (C2f) 的输入端
✅ 10% 结构化剪枝模型已保存: /content/drive/MyDrive/Colab Notebooks/DSAIproject/yolov8n_struct_pruned_10.pt


## 3.2.2 指定层位联动结构化剪枝30%

In [ ]:
import torch
import torch.nn as nn
from ultralytics import YOLO
import os
from copy import deepcopy

def create_pruned_conv(conv_layer, keep_indices, mode='out'):
    if mode == 'out':
        new_conv = nn.Conv2d(conv_layer.in_channels, len(keep_indices), conv_layer.kernel_size,
                             conv_layer.stride, conv_layer.padding, bias=conv_layer.bias is not None)
        new_conv.weight.data = conv_layer.weight.data[keep_indices].clone()
        if conv_layer.bias is not None: new_conv.bias.data = conv_layer.bias.data[keep_indices].clone()
    else:
        new_conv = nn.Conv2d(len(keep_indices), conv_layer.out_channels, conv_layer.kernel_size,
                             conv_layer.stride, conv_layer.padding, bias=conv_layer.bias is not None)
        new_conv.weight.data = conv_layer.weight.data[:, keep_indices].clone()
        if conv_layer.bias is not None: new_conv.bias.data = conv_layer.bias.data.clone()
    return new_conv

def create_pruned_bn(bn_layer, keep_indices):
    new_bn = nn.BatchNorm2d(len(keep_indices), eps=bn_layer.eps, momentum=bn_layer.momentum)
    if bn_layer.affine:
        new_bn.weight.data = bn_layer.weight.data[keep_indices].clone()
        new_bn.bias.data = bn_layer.bias.data[keep_indices].clone()
    return new_bn

# 2. 执行剪枝逻辑
finetune_ckpt = "/content/drive/MyDrive/Colab Notebooks/DSAIproject/yolov8n_finetune.pt"
model = YOLO(finetune_ckpt)
m = model.model
PRUNING_RATIO = 0.30

# 策略：只选择 Backbone 中路径清晰的卷积层 (1, 2, 4, 6)
# 避开具有 Concat 结构的深层模块，确保评估不报错
target_layers = [1, 2, 4, 6]

print(f"✂️ 正在对层 {target_layers} 执行 30% 结构化剪枝...")

for i in target_layers:
    layer = m.model[i]
    if hasattr(layer, 'conv') and hasattr(layer, 'bn'):
        total = layer.bn.num_features
        keep_count = int(total * (1 - PRUNING_RATIO))
        keep_idx = list(range(keep_count))

        # 剪当前层
        layer.conv = create_pruned_conv(layer.conv, keep_idx, mode='out')
        layer.bn = create_pruned_bn(layer.bn, keep_idx)

        # 联动修改下一层 (i+1) 的输入口
        next_layer = m.model[i+1]
        if hasattr(next_layer, 'conv'):
            next_layer.conv = create_pruned_conv(next_layer.conv, keep_idx, mode='in')
        elif hasattr(next_layer, 'cv1'): # 如果是 C2f 模块
            next_layer.cv1.conv = create_pruned_conv(next_layer.cv1.conv, keep_idx, mode='in')

# 3. 保存模型
save_path_30 = "/content/drive/MyDrive/Colab Notebooks/DSAIproject/yolov8n_struct_pruned_30.pt"
model.save(save_path_30)
print(f"✅ 30% 结构化模型已生成并保存：{save_path_30}")

✂️ 正在对层 [1, 2, 4, 6] 执行 30% 结构化剪枝...
✅ 30% 结构化模型已生成并保存：/content/drive/MyDrive/Colab Notebooks/DSAIproject/yolov8n_struct_pruned_30.pt


In [ ]:
import os
s1 = os.path.getsize("/content/drive/MyDrive/Colab Notebooks/DSAIproject/yolov8n_finetune.pt")/ (1024 * 1024)
s2 = os.path.getsize("/content/drive/MyDrive/Colab Notebooks/DSAIproject/yolov8n_finetune_pruned_10.pt")/ (1024 * 1024)
s3 = os.path.getsize("/content/drive/MyDrive/Colab Notebooks/DSAIproject/yolov8n_finetune_pruned_30.pt")/ (1024 * 1024)
s4 = os.path.getsize("/content/drive/MyDrive/Colab Notebooks/DSAIproject/yolov8n_struct_pruned_10.pt")/ (1024 * 1024)
s5 = os.path.getsize("/content/drive/MyDrive/Colab Notebooks/DSAIproject/yolov8n_struct_pruned_30.pt")/ (1024 * 1024)
print(f"s1: {s1}, s2: {s2}, s3: {s3},s4: {s4}, s5: {s5}")


s1: 5.958841323852539, s2: 5.981819152832031, s3: 5.981819152832031,s4: 5.980756759643555, s5: 5.978498458862305


## 3.3 宽度缩放-结构化剪枝



## 3.3.1 宽度缩放-结构化剪枝10% + Finetune10轮

In [ ]:
import torch
import yaml
from ultralytics import YOLO
import os

def structural_pruning_10_fixed(source_path, save_path):
    # 1. 加载原始微调模型获取权重
    old_model = YOLO(source_path)
    old_state = old_model.model.state_dict()

    # 2. 获取原始配置并修改宽度系数 (0.25 * 0.9 = 0.225)
    temp_yaml = "yolov8n_pruned_10.yaml"
    config = old_model.model.yaml
    config['width_multiple'] = 0.225 # 修改此处：从 0.175 改为 0.225

    with open(temp_yaml, 'w') as f:
        yaml.dump(config, f)

    # 3. 初始化瘦身后的空模型
    pruned_model = YOLO(temp_yaml)
    new_state = pruned_model.model.state_dict()

    print("✂️ 正在执行 10% 权重迁移与物理切片...")

    # 4. 权重迁移：按索引切片拷贝
    for key in new_state.keys():
        if key in old_state:
            old_w = old_state[key]
            new_w = new_state[key]

            if old_w.shape != new_w.shape:
                if len(old_w.shape) == 4: # 卷积核 [Out, In, K, K]
                    new_state[key] = old_w[:new_w.shape[0], :new_w.shape[1], :, :].clone()
                elif len(old_w.shape) == 1: # Bias 或 BN 参数
                    new_state[key] = old_w[:new_w.shape[0]].clone()
            else:
                new_state[key] = old_w.clone()

    # 5. 加载权重并保存最终模型
    pruned_model.model.load_state_dict(new_state)
    pruned_model.save(save_path)

    if os.path.exists(temp_yaml):
        os.remove(temp_yaml)

    # 验证
    old_params = sum(p.numel() for p in old_model.model.parameters())
    new_params = sum(p.numel() for p in pruned_model.model.parameters())
    print(f"✅ 成功生成 10% 剪枝模型！")
    print(f"📊 原始参数: {old_params} -> 剪枝后: {new_params}")
    print(f"💾 模型已保存至: {save_path}")

# --- 运行修改后的路径 ---
source_ckpt = "/content/drive/MyDrive/Colab Notebooks/DSAIproject/yolov8n_finetune.pt"
target_ckpt = "/content/drive/MyDrive/Colab Notebooks/DSAIproject/yolov8n_struct_pruned_10_new.pt"

structural_pruning_10_fixed(source_ckpt, target_ckpt)

✂️ 正在执行 10% 权重迁移与物理切片...
✅ 成功生成 10% 剪枝模型！
📊 原始参数: 3012993 -> 剪枝后: 2653001
💾 模型已保存至: /content/drive/MyDrive/Colab Notebooks/DSAIproject/yolov8n_struct_pruned_10_new.pt


In [ ]:
import os
import yaml
from ultralytics import YOLO

# --- 1. 配置基础路径 (根据你的环境) ---
dataset_root = "/content/dataset"
yaml_path = os.path.join(dataset_root, "data.yaml")
images_dir = "export/images"  # 你的图片实际存放位置

# --- 2. 修正 data.yaml 路径 ---
if os.path.exists(yaml_path):
    with open(yaml_path, 'r') as f:
        config = yaml.safe_load(f)

    # 强制将所有路径指向 export/images，确保训练能找到图片
    config['path'] = dataset_root
    config['train'] = images_dir
    config['val'] = images_dir
    config['test'] = images_dir

    with open(yaml_path, 'w') as f:
        yaml.dump(config, f, sort_keys=False)

    print("✅ data.yaml 路径已修正，全部指向 export/images")
else:
    print(f"❌ 错误：找不到 {yaml_path}")

# --- 3. 加载物理剪枝后的模型并微调 ---
# 这里的输入是你之前生成的“手术后”模型
pruned_model_input = "/content/drive/MyDrive/Colab Notebooks/DSAIproject/yolov8n_struct_pruned_10_new.pt"
# 这里的输出是你要求的“回血后”模型名
finetune_model_output = "/content/drive/MyDrive/Colab Notebooks/DSAIproject/yolov8n_struct_pruned_10_finetune.pt"

if os.path.exists(pruned_model_input):
    print(f"🚀 正在加载剪枝模型: {pruned_model_input}")
    model = YOLO(pruned_model_input)

    print("开始 10 轮微调训练...")
    # 启动训练
    model.train(
        data=yaml_path,
        epochs=10,
        imgsz=640,
        batch=16,
        lr0=0.001,      # 学习率设小，稳定残余权重
        warmup_epochs=2, # 热身轮次，关键：校准 BN 层以恢复 mAP
        device=0,
        project="Struct_Pruning_Project",
        name="finetune_run",
        exist_ok=True
    )

    # --- 4. 保存最终模型 ---
    model.save(finetune_model_output)
    print(f"🎉 微调完成！模型已保存至: {finetune_model_output}")
else:
    print(f"❌ 错误：找不到剪枝后的基础模型文件 {pruned_model_input}")

✅ data.yaml 路径已修正，全部指向 export/images
🚀 正在加载剪枝模型: /content/drive/MyDrive/Colab Notebooks/DSAIproject/yolov8n_struct_pruned_10_new.pt
开始 10 轮微调训练...
Ultralytics 8.4.25 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/Colab Notebo


## 3.3.2 宽度缩放-结构化剪枝30%+Finetune10轮



In [ ]:
import torch
import yaml
from ultralytics import YOLO
import os

def structural_pruning_30_fixed(source_path, save_path):
    # 1. 加载原始微调模型获取权重
    old_model = YOLO(source_path)
    old_state = old_model.model.state_dict()

    # 2. 获取原始配置并修改宽度系数 (0.25 * 0.7 = 0.175)
    # 我们需要先获取 yolov8n 的配置字典
    temp_yaml = "yolov8n_pruned_30.yaml"
    config = old_model.model.yaml
    config['width_multiple'] = 0.175

    # 将字典保存为真实的 YAML 文件，避免 "File name too long" 错误
    with open(temp_yaml, 'w') as f:
        yaml.dump(config, f)

    print(f"📦 已创建剪枝架构配置文件: {temp_yaml}")

    # 3. 初始化瘦身后的空模型
    # 现在传入的是文件路径，不会再报错
    pruned_model = YOLO(temp_yaml)
    new_state = pruned_model.model.state_dict()

    print("✂️ 正在执行权重迁移与物理切片...")

    # 4. 权重迁移：按索引切片拷贝
    updated_keys = 0
    for key in new_state.keys():
        if key in old_state:
            old_w = old_state[key]
            new_w = new_state[key]

            # 物理切片逻辑
            if old_w.shape != new_w.shape:
                if len(old_w.shape) == 4: # 卷积核 [Out, In, K, K]
                    new_state[key] = old_w[:new_w.shape[0], :new_w.shape[1], :, :].clone()
                elif len(old_w.shape) == 1: # Bias 或 BN 参数
                    new_state[key] = old_w[:new_w.shape[0]].clone()
            else:
                new_state[key] = old_w.clone()
            updated_keys += 1

    # 5. 加载权重并保存最终模型
    pruned_model.model.load_state_dict(new_state)
    pruned_model.save(save_path)

    # 清理临时文件
    if os.path.exists(temp_yaml):
        os.remove(temp_yaml)

    # 验证
    old_params = sum(p.numel() for p in old_model.model.parameters())
    new_params = sum(p.numel() for p in pruned_model.model.parameters())
    print(f"✅ 成功生成 30% 剪枝模型！")
    print(f"📊 原始参数: {old_params} -> 剪枝后: {new_params}")
    print(f"💾 模型已保存至: {save_path}")

# --- 运行 ---
source_ckpt = "/content/drive/MyDrive/Colab Notebooks/DSAIproject/yolov8n_finetune.pt"
target_ckpt = "/content/drive/MyDrive/Colab Notebooks/DSAIproject/yolov8n_struct_pruned_30_new.pt"

structural_pruning_30_fixed(source_ckpt, target_ckpt)

📦 已创建剪枝架构配置文件: yolov8n_pruned_30.yaml
✂️ 正在执行权重迁移与物理切片...
✅ 成功生成 30% 剪枝模型！
📊 原始参数: 3012993 -> 剪枝后: 1734377
💾 模型已保存至: /content/drive/MyDrive/Colab Notebooks/DSAIproject/yolov8n_struct_pruned_30_new.pt


In [ ]:
# Finetune10 轮适应10轮分类任务
import os
import yaml
from ultralytics import YOLO

# --- 1. 配置基础路径 (根据你的环境) ---
dataset_root = "/content/dataset"
yaml_path = os.path.join(dataset_root, "data.yaml")
images_dir = "export/images"  # 你的图片实际存放位置

# --- 2. 修正 data.yaml 路径 ---
if os.path.exists(yaml_path):
    with open(yaml_path, 'r') as f:
        config = yaml.safe_load(f)

    # 强制将所有路径指向 export/images，确保训练能找到图片
    config['path'] = dataset_root
    config['train'] = images_dir
    config['val'] = images_dir
    config['test'] = images_dir

    with open(yaml_path, 'w') as f:
        yaml.dump(config, f, sort_keys=False)

    print("✅ data.yaml 路径已修正，全部指向 export/images")
else:
    print(f"❌ 错误：找不到 {yaml_path}")

# --- 3. 加载物理剪枝后的模型并微调 ---
# 这里的输入是你之前生成的“手术后”模型
pruned_model_input = "/content/drive/MyDrive/Colab Notebooks/DSAIproject/yolov8n_struct_pruned_30_new.pt"
# 这里的输出是你要求的“回血后”模型名
finetune_model_output = "/content/drive/MyDrive/Colab Notebooks/DSAIproject/yolov8n_struct_pruned_30_finetune.pt"

if os.path.exists(pruned_model_input):
    print(f"🚀 正在加载剪枝模型: {pruned_model_input}")
    model = YOLO(pruned_model_input)

    print("开始 10 轮微调训练...")
    # 启动训练
    model.train(
        data=yaml_path,
        epochs=10,
        imgsz=640,
        batch=16,
        lr0=0.001,      # 学习率设小，稳定残余权重
        warmup_epochs=2, # 热身轮次，关键：校准 BN 层以恢复 mAP
        device=0,
        project="Struct_Pruning_Project",
        name="finetune_run",
        exist_ok=True
    )

    # --- 4. 保存最终模型 ---
    model.save(finetune_model_output)
    print(f"🎉 微调完成！模型已保存至: {finetune_model_output}")
else:
    print(f"❌ 错误：找不到剪枝后的基础模型文件 {pruned_model_input}")

✅ data.yaml 路径已修正，全部指向 export/images
🚀 正在加载剪枝模型: /content/drive/MyDrive/Colab Notebooks/DSAIproject/yolov8n_struct_pruned_30_new.pt
开始 10 轮微调训练...
Ultralytics 8.4.25 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/Colab Notebo



##  4.定义评估函数，批量评估各个模型性能

* 01_Baseline_Pretrained path:yolov8n.pt 原始官方版本

* 02_Finetuned_Full path:yolov8n_finetune.pt 微调10轮适配11类任务

* 03_Finetuned_Unstructured_10 path:yolov8n_finetune_pruned_10.pt 在02基础上非结构化剪枝10%

* 04_Finetuned_Unstructured_30 path:yolov8n_finetune_pruned_30.pt 在02基础上非结构化剪枝30%

* 05_Finetuned_Unstructured_50 path:yolov8n_finetune_pruned_50.pt 在02基础上非结构化剪枝50%

* 06_Finetuned_Structured_10 path:yolov8n_struct_pruned_10.pt 在02基础上结构化剪枝10%

* 07_Finetuned_Structured_30 path:yolov8n_struct_pruned_30.pt 在02基础上结构化剪枝30%

* 08_Finetuned_Structured_10_phys path:yolov8n_struct_pruned_10_new.pt 在02基础上结构化剪枝+物理缩小10%

* 09_Finetuned_Structured_10_phys_ft path:yolov8n_struct_pruned_10_finetune.pt 在02基础上结构化剪枝+物理缩小10%+微调10轮

* 10_Finetuned_Structured_30_phys path:yolov8n_struct_pruned_30_new.pt 在02基础上结构化剪枝+物理缩小30%

* 11_Finetuned_Structured_30_phys_ft path:yolov8n_struct_pruned_30_finetune.pt 在02基础上结构化剪枝+物理缩小30%+微调10轮



In [ ]:
## 定义评估函数
from datetime import datetime
PROJECT_DIR = '/content/drive/MyDrive/Colab Notebooks/DSAIproject'

RECORD_CSV_PATH = os.path.join(PROJECT_DIR, 'test_record.csv')

def evaluate_and_record(model_path, data_yaml, version_name, save_path=RECORD_CSV_PATH, device='cuda'):
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    model = YOLO(model_path)

    # 【校验核心】检查权重校验和
    first_weight_sum = next(model.model.parameters()).sum().item()
    print(f"DEBUG: [{version_name}] 权重校验和: {first_weight_sum:.6f}")

    file_size = os.path.getsize(model_path) / (1024 * 1024)
    params = sum(p.numel() for p in model.model.parameters())

    # 1. 运行验证
    results = model.val(data=data_yaml, device=device, plots=False, verbose=False)
    precision, recall = results.box.mp, results.box.mr
    map50, map50_95 = results.box.map50, results.box.map
    f1_score = 2 * (precision * recall) / (precision + recall + 1e-9)

    # 2. 推理速度测试（优化：使用 rand 消除警告 + 硬件同步）
    dummy_input = torch.rand((1, 3, 640, 640)).to(device)
    with torch.no_grad():
        for _ in range(30): _ = model(dummy_input, device=device, verbose=False)
        if 'cuda' in str(device): torch.cuda.synchronize()
        start_time = time.time()
        for _ in range(100): _ = model(dummy_input, device=device, verbose=False)
        if 'cuda' in str(device): torch.cuda.synchronize()
        avg_latency = ((time.time() - start_time) / 100) * 1000
    fps = 1000 / avg_latency

    # 3. 数据汇总
    try: flops = model.info(verbose=False)[4] or 0
    except: flops = 0

    new_record = {
        "Version": version_name,
        "Model_File": os.path.basename(model_path),
        "P": round(float(precision), 4), "R": round(float(recall), 4), "F1": round(float(f1_score), 4),
        "mAP50": round(float(map50), 4), "mAP50-95": round(float(map50_95), 4),
        "Size_MB": round(float(file_size), 2), "Params": int(params), "GFLOPs": round(float(flops), 2),
        "Latency_ms": round(float(avg_latency), 2), "FPS": round(float(fps), 2),
        "Date": datetime.now().strftime("%Y-%m-%d %H:%M:%S")  # <-- 记录时间
    }

    # 4. 打印与保存
    print(f"\n{'='*20} {version_name}: F1={new_record['F1']}, FPS={new_record['FPS']} {'='*20}")
    df = pd.DataFrame([new_record])
    df.to_csv(save_path, mode='a', header=not os.path.exists(save_path), index=False)

    return new_record

In [ ]:
import os
import torch
import time
import pandas as pd
import numpy as np
import yaml


!mkdir -p /content/dataset/valid
if not os.path.exists('/content/dataset/valid/images'):
    # 将实际的 images 目录 映射到 模型想找的 valid/images
    !ln -s /content/dataset/export/images /content/dataset/valid/images
    print("✅ 路径软链接已创建，Baseline 评估应能通过。")

# --- 1. 定义模型路径清单 ---
base_path = "/content/drive/MyDrive/Colab Notebooks/DSAIproject"
models_to_test = [
    {"name": "01_Baseline_Pretrained", "path": "yolov8n.pt"},
    {"name": "02_Finetuned_Full",      "path": "yolov8n_finetune.pt"},
    {"name": "03_Finetuned_Unstructured_10",     "path": "yolov8n_finetune_pruned_10.pt"},
    {"name": "04_Finetuned_Unstructured_30",     "path": "yolov8n_finetune_pruned_30.pt"},
    {"name": "05_Finetuned_Unstructured_50",     "path": "yolov8n_finetune_pruned_50.pt"},
    {"name": "06_Finetuned_Structured_10",       "path": "yolov8n_struct_pruned_10.pt"},
    {"name": "07_Finetuned_Structured_30",       "path": "yolov8n_struct_pruned_30.pt"},
    {"name": "08_Finetuned_Structured_10_phys",       "path": "yolov8n_struct_pruned_10_new.pt"},
    {"name": "09_Finetuned_Structured_10_phys_ft",       "path": "yolov8n_struct_pruned_10_finetune.pt"},
    {"name": "10_Finetuned_Structured_30_phys",       "path": "yolov8n_struct_pruned_30_new.pt"},
    {"name": "11_Finetuned_Structured_30_phys_ft",       "path": "yolov8n_struct_pruned_30_finetune.pt"},

]

# --- 2. 自动化循环评估 ---
print(f"🕵️ 开始全模型对比评估，结果将记录至: {RECORD_CSV_PATH}\n")

for m_info in models_to_test:
    full_path = os.path.join(base_path, m_info["path"])

    if os.path.exists(full_path):
        print(f"🔄 正在评估: {m_info['name']} ...")
        try:
            # 调用优化后的评估函数
            evaluate_and_record(
                model_path=full_path,
                data_yaml=yaml_path,
                version_name=m_info["name"],
                device='cuda' if torch.cuda.is_available() else 'cpu'
            )
        except Exception as e:
            print(f"❌ {m_info['name']} 评估失败: {e}")
    else:
        print(f"⚠️ 跳过: 找不到文件 {full_path}")

print("\n✅ 所有模型评估完成！请查看 test_record.csv 或上方打印的报告。")

🕵️ 开始全模型对比评估，结果将记录至: /content/drive/MyDrive/Colab Notebooks/DSAIproject/test_record.csv

🔄 正在评估: 05_Finetuned_Unstructured_50 ...
DEBUG: [05_Finetuned_Unstructured_50] 权重校验和: -58.734375
Ultralytics 8.4.26 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,007,793 parameters, 0 gradients
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 938.2±299.5 MB/s, size: 34.0 KB)
val: Scanning /content/dataset/export/labels.cache... 29800 images, 3500 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 29800/29800 7.4Git/s 0.0s
val: /content/dataset/export/images/1478021875081281646_jpg.rf.bEZPhuyXU5hIovwQSTIp.jpg: 1 duplicate labels removed
val: /content/dataset/export/images/1478021875081281646_jpg.rf.e9552980cf8c6fef4aa02cb84c6364f5.jpg: 1 duplicate labels removed
val: /content/dataset/export/images/1478897760163798179_jpg.rf.5Pzrj3Eg3vZuyl7ztKAt.jpg: 1 duplicate labels removed
val: /content/dataset/export/images/1478897760163798179_jpg.rf.98623be50b02ff17